# VLA Workshop — robosuite × SmolVLA

Google Colabで、環境確認からVLA推論まで順に取り組みます。実習プログラムをこのipynbに同梱しています。教材の.pyやGitHubリポジトリを別途取得する必要はありません。
ライブラリと学習済みモデルは実行時にダウンロードします。

1. ランタイム → ランタイムのタイプを変更 → **T4 GPU**を選びます。ノートブックにもT4を要求する設定を入れています。
2. このノートブックを自分のDriveにコピーし、上から順番に実行します。
3. 最初は00と01を各20ステップだけ試します。タスク成功は今回の合格条件ではありません。
4. 最後に動画・ログを自分のDriveへ保存し、ランタイムを接続解除して削除します。

Python 3.12／3.13向けです。初回セットアップとモデル取得には数分以上かかる場合があります。

接続そのものが失敗する場合は、この教材のコードはまだ実行されていません。
「T4 GPU」を選択し、Colabへの接続が完了してからセルを実行してください。
T4の空きや利用枠によっては接続できないことがあり、ノートブックから割り当てを強制することはできません。


In [ ]:
# T4とPythonを確認する（Colabへの接続完了後に実行）
import subprocess
import sys

print(sys.version)
if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Python 3.12または3.13のランタイムを選んでください")
try:
    gpu_names = (
        subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
        )
        .strip()
        .splitlines()
    )
except (FileNotFoundError, subprocess.CalledProcessError) as error:
    raise RuntimeError(
        "ランタイムのタイプをT4 GPUに変更し、接続してから実行してください"
    ) from error
print("割り当てられたGPU:", gpu_names)
if len(gpu_names) != 1 or "T4" not in gpu_names[0].upper():
    raise RuntimeError("ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください")

## 内蔵プログラムを準備する

このセルを実行すると、ipynbに同梱した実習コードをColab内に準備します。設定の変更は不要です。
教材リポジトリのcloneや、教材の.pyを別途ダウンロードする操作はありません。

ノートブックは共有Driveから開いても、GitHubの「Open in Colab」から開いても同じ内容です。
「ファイル → ドライブにコピーを保存」から自分用のコピーを作り、上から順に実行してください。


In [ ]:
import json
from pathlib import Path

PROJECT = Path("/content/vla_workshop")
raise RuntimeError("講師用テンプレートです。生成済みの実習ノートブックを開いてください。")

## 共通環境をセットアップする

Pythonの版に対応する固定依存リストから、教材専用の環境を作ります。
Colab標準ライブラリの更新に影響されないよう、この環境のPythonから実習を実行します。
CUDA 12.8版のPyTorchと対応するtorchvisionを使います。

初回はCUDA関連パッケージも取得するため時間がかかります。進捗はセルと`outputs/setup.log`に表示します。
インストール後に`pip check`と主要ライブラリのimport、T4上のCUDA計算を確認してから次へ進みます。


In [ ]:
# @title 実習用の環境を準備する
subprocess.run(
    [sys.executable, str(PROJECT / "scripts/setup_colab.py"), "--project", str(PROJECT)], check=True
)
PYTHON = PROJECT / ".venv/bin/python"
print("実習用Python:", PYTHON)


def run_lesson(script, *, steps=20, episodes=1):
    result = subprocess.run(
        [
            str(PYTHON),
            str(PROJECT / script),
            "--config",
            str(PROJECT / "configs/lift.json"),
            "--max-steps",
            str(steps),
            "--episodes",
            str(episodes),
        ],
        cwd=PROJECT,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr[-6000:])
    result.check_returncode()
    line = next(line for line in result.stdout.splitlines() if line.startswith("Results: "))
    return Path(line.removeprefix("Results: "))

## 00: 環境と観測を確認する

標準Lift環境にPandaを置きます。画像・状態・制御・動画保存を確認します。
この段階は学習モデルを使わず、手先の移動指令をゼロ、グリッパーを開く指令にします。

In [ ]:
inspect_result = run_lesson("lessons/00_environment/run.py")

In [ ]:
from IPython.display import Image, Video, display

display(Image(filename=str(inspect_result / "episode_000/agentview.png")))
display(Video(str(inspect_result / "episode_000/video.mp4"), embed=True))
print((inspect_result / "summary.json").read_text())
print((inspect_result / "episode_000/environment.json").read_text())

### 確認課題

- 画像にはPandaと立方体が写っていますか。
- 8次元の状態と7次元の行動について、各要素の意味を説明できますか。
- 20 Hzという設定と、このセルの実行時間の違いを説明できますか。

詳しい仕様は教材内の`docs/io_contract.md`、読むコードは`src/vla_workshop/envs/robosuite_env.py`です。

## 01: CUDAでSmolVLAを推論する

LIBERO用チェックポイントの画像2枚・8次元状態・7次元行動に合わせて接続します。
モデル付属の正規化・逆正規化を使い、実行後は再び観測して次の行動を選びます。

この実習のシーンはLiftです。ここでの成否をLIBEROの成功率として扱わないでください。

In [ ]:
MAX_STEPS = 20
EPISODES = 1

inference_result = run_lesson(
    "lessons/01_smolvla_inference/run.py", steps=MAX_STEPS, episodes=EPISODES
)

In [ ]:
from IPython.display import Video, display

display(Video(str(inference_result / "episode_000/video.mp4"), embed=True))
print((inference_result / "summary.json").read_text())
run_info = json.loads((inference_result / "run.json").read_text())
print(json.dumps(run_info["policy"], indent=2, ensure_ascii=False))
steps = [
    json.loads(line)
    for line in (inference_result / "episode_000/steps.jsonl").read_text().splitlines()
]
print("最初のstep:", json.dumps(steps[0], indent=2, ensure_ascii=False))

### 確認課題

- 実際のデバイスが`cuda:0`になっていますか。
- モデルの行動と環境に渡した行動は一致していますか。clipされた場合はどの要素ですか。
- 失敗した場合、動画から何が起きたか説明できますか。
- 20ステップの確認後、`MAX_STEPS=200`に変えて試してください。`EPISODES=3`ではseedも変わります。

今回の合格条件は「観測→推論→行動実行を通し、動画とログを保存し、内容を説明できること」です。

## 結果をDriveへ保存して終了する

動画・ログ・実行条件をまとめて、自分のDriveに保存します。Google Driveへの接続承認が必要です。
`RESULTS_DIR`を変更すれば、共有フォルダ内の自分の結果フォルダにも保存できます。
共有フォルダを使う場合は、自分のMyDriveにそのフォルダのショートカットを追加してパスを指定してください。

保存後、ランタイムの「接続解除して削除」を実行します。


In [ ]:
# @title 結果をDriveへ保存する
import shutil
from datetime import UTC, datetime
from uuid import uuid4

from google.colab import drive

RESULTS_DIR = "/content/drive/MyDrive/vla_Workshop/results"  # @param {type:"string"}
drive.mount("/content/drive")
destination = Path(RESULTS_DIR)
destination.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
name = f"LESSON_results_{timestamp}_{uuid4().hex[:8]}"
archive = shutil.make_archive("/content/" + name, "zip", PROJECT / "outputs")
saved = shutil.copy2(archive, destination / (name + ".zip"))
print("Driveに保存しました:", saved)